# Download files

Download all eval Excel files.
* ACCESS Advising
* SOC Advising
* ACCESS MPA Peer Advising

## All Steps: Generate summary data...with functions - MPA 😎

Generate summary data as html tables
* create summary for Lichert scale responses (i.e., pivot table of responses)
* get comments
* split topics (comma-separated responses)
* combine summary data

### Advisors who need a date range: 
modify 
* Cell 1: time variables: filter_year and filter_month
* Cell 4: df = df[df['Timestamp'].dt.year == filter_year]
    df = df[df['Timestamp'].dt.month == filter_month]
* Cell 12: headers

## 1. Set variables

In [1]:
# files to open
access_eval = 'ACCESS Advising Evaluation (Responses).xlsx'
soc_eval = 'SOC Advising (Responses).xlsx'
mpa_eval = 'ACCESS MPA Peer Advising – Student Feedback Survey (Responses).xlsx'

# time variables
filter_year = 2026 # for date range: enter the starting year
filter_month = 11 # single month: enter the exact month ex: 11 for November

# questions to remove
# November 2025 survey questions changed and Google Sheets was reindexed.
# Starting December 2025, questions do not need to be removed.

# text to remove from each question
eval_remove_text = ['Please evaluate your advising session',
                    'Please evaluate the advisor',
                    'Overall advising appointment',
                    'Please rate the following',
                    '[',
                    ']',
                    '.',
                    ':',
                    '1', '2', '3', '4',
                    'Check all that apply.']

# text to replace
comments_replace = ['Please feel free to make additional comments of the above questions',
                    'Additional comments?',
                    'Any additional comments?']

# Likert scale values
likert_scale_access1 = ['Strongly Agree (5)', 'Agree (4)', 'Neutral (3)', 'Disagree (2)', 'Strongly Disagree (1)', 'Not Applicable (0)', 'No Response']
likert_scale_access2 = ['Excellent (5)', 'Good (4)', 'Average (3)', 'Fair (2)', 'Poor (1)', 'No Response']
likert_scale_soc1 = ['Strongly agree', 'Agree', 'Neutral', 'Disagree', 'Strongly Disagree', 'No Response']
likert_scale_soc2 = ['Excellent', 'Good', 'Average', 'Fair', 'Poor', 'No Response']
likert_scale_mpa1 = ['Strongly Agree', 'Agree', 'Neutral', 'Disagree', 'Strongly Disagree', 'N/A', 'No Response']
likert_scale_mpa2 = ['Excellent', 'Good', 'Average', 'Fair', 'Poor', 'No Response']

# meeting types
meeting_types = ['Via email', 'Via phone', 'Via Zoom', 'In person']

## 2. Load and clean data (from Step 1)

In [2]:
# load libraries
import pandas as pd
from datetime import datetime
from IPython.display import display, HTML  # library to display html in notebook
from pathlib import Path

In [3]:
# load data
folder_path = Path(r"C:\Users\Susan\Downloads\evals")
access_df = pd.read_excel(f'{folder_path}/{access_eval}')
soc_df = pd.read_excel(f'{folder_path}/{soc_eval}')
mpa_df = pd.read_excel(f'{folder_path}/{mpa_eval}')

In [4]:
# clean data
def clean_data(df, text_remove_list, text_replace_list):
    for i in text_remove_list:
        df.columns = df.columns.str.replace(i, '')

    for i in text_replace_list:
        df.columns = df.columns.str.replace(i, 'Comments')

    df.columns = df.columns.str.strip()

    # single month
    df = df[df['Timestamp'].dt.year == filter_year]
    df = df[df['Timestamp'].dt.month == filter_month]

    # date range
    #df = df[df['Timestamp'].dt.year >= filter_year]
    #df = df[df['Timestamp'].dt.month >= filter_month]
    
    df = df.fillna('No Response')
    
    return df

# clean data
access_df = clean_data(access_df, eval_remove_text, comments_replace)
soc_df = clean_data(soc_df, eval_remove_text, comments_replace)
mpa_df = clean_data(mpa_df, eval_remove_text, comments_replace)

In [5]:
# check data
# mpa_df

## 3. Filter data for a single advisor

In [6]:
# filter data for a single advisor

# original
#def filtered_advisor_df (advisor_name, df_name):
#    filtered_advisor = df_name[df_name['Advisor'].str.startswith(advisor_name)]
    
#    return filtered_advisor

# Include MPA
def filtered_advisor_df (advisor_name, df_name, adv_type):
    if adv_type == 'mpa':
        filtered_advisor = df_name[df_name['Which Manoa Peer Advisor (MPA) did you meet with?'].str.startswith(advisor_name)]
    else:
        filtered_advisor = df_name[df_name['Advisor'].str.startswith(advisor_name)]
    return filtered_advisor


# testing
#advisor_list = mpa_df['Which Manoa Peer Advisor (MPA) did you meet with?'].unique()
#advisor_name = advisor_list[0]
#df_name = mpa_df

#test1 = df_name[df_name['Which Manoa Peer Advisor (MPA) did you meet with?'].str.startswith(advisor_name[0])]
#test1 = filtered_advisor_df (advisor_name, df_name, 'mpa')
#test1

In [7]:
# check data
#filtered_advisor
#filtertest

## 4. Functions to generate summary data

In [8]:
# function to create summary data for Likert scale responses
# def function_name(var1, var2):
#    code to create summary data
#    return html_table

def create_summary_data(col_beg, col_end, scale, filtered_advisor, show_percent=True):
    question_score = []

    # get list of questions
    questions = []
    questions = filtered_advisor.columns[col_beg:col_end]

    number_responses = len(filtered_advisor)

    for i in questions:
        for j in scale:
            count = (filtered_advisor[i] == j).sum()
            percent = count / number_responses * 100
            display_count = f'{count}<br>{percent:.1f}%'

            if show_percent == False:
                display_count = count
                
            question_score.append({
                'Question': i,
                'Score': j,
                'Count': display_count
            })

    df = pd.DataFrame(question_score)
    df = df.pivot(index='Question', columns='Score', values='Count')

    df = df[scale].loc[questions]
    df.columns.name = None

    html_likert = df.to_html(escape=False, classes='table table-striped table-bordered text-left')
    
    return html_likert

In [9]:
# function to get comments
# def function_name(var1, var2):
#    code to get comments
#    return html_table

def create_comments_data(col_beg, col_end, filtered_advisor, advtype):
    comments = filtered_advisor.iloc[:, col_end]
    
    comments = comments[comments != 'No Response']

    if advtype == 'mpa1':
        commentsdf = pd.DataFrame({'What was most helpful about your Peer Advising session?': comments})
    elif advtype == 'mpa2':
        commentsdf = pd.DataFrame({'What could be improved?': comments})
    else:
        commentsdf = pd.DataFrame({'Comments': comments})
    
    html_comments = commentsdf.to_html(index=False, classes='table table-striped table-bordered text-left')

    if len(comments) == 0:
        html_comments = ''

    return html_comments

# testing
#advisor_list = mpa_df['Which Manoa Peer Advisor (MPA) did you meet with?'].unique()
#advisor_name = advisor_list[0]
#df_name = mpa_df

#test1 = df_name[df_name['Which Manoa Peer Advisor (MPA) did you meet with?'].str.startswith(advisor_name[0])]
#test1 = filtered_advisor_df (advisor_name, df_name, 'mpa')
#test2 = create_comments_data(0, -3, test1)
#test2

In [10]:
# function to split comma-separated responses
# def function_name(var1, var2):
#    code to split responses
#    return html_table

def soc_split_topics(meeting_col, rename_col, filtered_advisor):
    soc_topics = []

    meeting = filtered_advisor.columns[meeting_col]
    responses = filtered_advisor[meeting].values

    for i in responses:
        # replace response that contains a comma
        i = i.replace('Major information, options', 'Major information/options')
        i = i.replace(', ', ',')

        # split values by comma
        split = i.split(',')

        # add topic to list
        for j in split:
            soc_topics.append(j)

    df = pd.DataFrame({f'{meeting}': soc_topics})

    summary = df[meeting].value_counts().reset_index()
    summary.columns = [meeting, 'Number of Responses*']

    html_topics = summary.to_html(index=False, classes='table table-striped table-bordered text-left')

    html_topics = html_topics + '<p style="font-size: 10px">* One student can select more than one topic during a meeting.</p>'

    return html_topics

In [11]:
# function to combine summary data (put html tables together)
# def function_name(var1, var2):
#    code to combine html tables together
#    return html_final
# html1 = html_likert
# html2 = html_comments
# html3 = html_topics
# html4 = html_meeting_types


def combine_summary_data(df_name, html1, html2, html3, html4):
    if df_name == 'soc_df':
        html_summary_data = html4 + html3 + html1 + html2
    else:
        html_summary_data = html1 + html2

    return html_summary_data

In [12]:
# call function

# loop through both lists, shift the columns for each type of likert scales
# loop for each advisor

html_file = ''
html_topics = ''
html_meeting_mode = ''
date = datetime(filter_year, filter_month, 1).strftime("%B %Y")
pagebreak = '<div style="page-break-before: always;"></div>'

# filtered_advisor = filtered_advisor_df ('Marilou', access_df)

html_header = f"""
    <!DOCTYPE html> 
        <html lang="en"> 
        <head>
            <title>Monthly Advisor Evals</title>
            <meta charset="utf-8">   
            <meta name="viewport" content="width=device-width, initial-scale=1">
            <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.8/dist/css/bootstrap.min.css" rel="stylesheet">
            <script src="https://cdn.jsdelivr.net/npm/bootstrap@5.3.8/dist/js/bootstrap.bundle.min.js"></script>
            <style>       
                body {{ margin: 20px; font-family: 'Verdana', sans-serif; }}
                table {{ width: auto; font-size: 13px}}
                th, td {{ text-align: left; font-size: 13px}}
            </style>
        </head>
        <body>
            <div class="container-xl">
"""
# access_df
advisors_list = access_df['Advisor'].unique()

for index, advisor in enumerate(advisors_list):
    filtered_advisor = filtered_advisor_df(advisor, access_df, '')

    html_header = html_header + '<h1>Monthly Evals for ' + advisor + '</h1>' + '<h5>Date: ' + date + '</h5>'
    
    # For date range
    #html_header = html_header + '<h1>Evals for ' + advisor + '</h1>' + '<h5>December 2022 - November 2026' + '</h5>'

    # likerts
    html_likert = create_summary_data(2, 9, likert_scale_access1, filtered_advisor)
    html_likert = html_likert + create_summary_data(9, 11, likert_scale_access2, filtered_advisor)

    # comments
    html_comments = create_comments_data(0, -1, filtered_advisor, '')

    # combined summary_data
    html_summary_data = combine_summary_data('access_df', html_likert, html_comments, html_topics, html_meeting_mode)

    # include page break
    if index < len(advisors_list) - 1:
        html_file = html_file + html_header + html_summary_data + pagebreak
    else:
        html_file = html_file + html_header + html_summary_data
    
    html_header = ''

# soc_df
advisors_list = soc_df['Advisor'].unique()

for index, advisor in enumerate(advisors_list):
    if len(html_file) > 0 and index == 0:
        html_file = html_file + pagebreak
        
    filtered_advisor = filtered_advisor_df(advisor, soc_df, '')

    html_header = '<h1>Monthly Evals for ' + advisor + '</h1>' + '<h5>Date: ' + date + '</h5>'
    
    # For date range
    #html_header = html_header + '<h1>Evals for ' + advisor + '</h1>' + '<h5>December 2022 - November 2026' + '</h5>'

    # likerts
    html_likert = create_summary_data(4, 6, likert_scale_soc1, filtered_advisor)
    html_likert = html_likert + create_summary_data(6, 8, likert_scale_soc2, filtered_advisor)

    # comments
    html_comments = create_comments_data(0, -1, filtered_advisor, '')

    # topics
    html_topics = soc_split_topics(3, 0, filtered_advisor)

    # meeting mode
    #html_meeting_mode = soc_meeting_mode(2, 0, filtered_advisor)
    html_meeting_mode = create_summary_data(2, 3, meeting_types, filtered_advisor, show_percent=False)

    # combined summary_data
    html_summary_data = combine_summary_data('soc_df', html_likert, html_comments, html_topics, html_meeting_mode)

    if index < len(advisors_list) - 1:
        html_file = html_file + html_header + html_summary_data + pagebreak
    else:
        html_file = html_file + html_header + html_summary_data
    
    html_header = ''
    html_topics = ''
    html_meeting_mode = ''

# mpa_df
advisors_list = mpa_df['Which Manoa Peer Advisor (MPA) did you meet with?'].unique()

for index, advisor in enumerate(advisors_list):
    if len(html_file) > 0 and index == 0:
        html_file = html_file + pagebreak
        
    filtered_advisor = filtered_advisor_df(advisor, mpa_df, 'mpa')

    html_header = html_header + '<h1>Monthly Evals for ' + advisor + '</h1>' + '<h5>Date: ' + date + '</h5>'

    # likerts
    html_likert = create_summary_data(2, 15, likert_scale_mpa1, filtered_advisor)
    html_likert = html_likert + create_summary_data(15, 17, likert_scale_mpa2, filtered_advisor)

    # comments
    html_comments = create_comments_data(0, -3, filtered_advisor, 'mpa1')
    html_comments = html_comments + create_comments_data(0, -2, filtered_advisor, 'mpa2')
    html_comments = html_comments + create_comments_data(0, -1, filtered_advisor, '')

    # combined summary_data
    html_summary_data = combine_summary_data('mpa_df', html_likert, html_comments, html_topics, html_meeting_mode)

    if index < len(advisors_list) - 1:
        html_file = html_file + html_header + html_summary_data + pagebreak
    else:
        html_file = html_file + html_header + html_summary_data
    
    html_header = ''

html_file = html_file + '</div></body></html>'
#display(HTML(html_file))

display(HTML(html_file))

# check data
file_path = 'eval.html'
with open(f'{folder_path}/{file_path}', 'w', encoding='utf-8') as file:
    file.write(html_file)

## 👏 You made really nice functions, yay!